# Practice 38 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — diamonds: translate to vectorized

Load `sns.load_dataset('diamonds')`. Rewrite each snippet without `.apply()`. No hints this time — you know these.

```python
# A
diamonds['value_score'] = diamonds.apply(
    lambda row: row['depth'] * row['table'] / row['price'], axis=1)

# B
diamonds['cut_len'] = diamonds['cut'].apply(lambda x: len(str(x)))

# C
diamonds['tier'] = diamonds['price'].apply(
    lambda x: 'budget' if x < 500
    else 'mid' if x < 2500
    else 'premium' if x < 10000
    else 'luxury')

# D
diamonds['log_price'] = diamonds['price'].apply(np.log)

# E
diamonds['heavy'] = diamonds['carat'].apply(lambda x: x > 1.5)
```

After rewriting: use `%timeit` to compare C (apply vs `np.select`). What is the speedup ratio?

In [ ]:
diamonds = sns.load_dataset('diamonds')

# Your rewrites here

#A
diamonds['value_score'] = diamonds['depth']*diamonds['table']/diamonds['price']

#B
diamonds['cut_len'] = diamonds['cut'].str.len()
#C
diamonds['tier'] = pd.cut(diamonds['price'], bins = [0,500,2500,10000,np.inf], labels=['budget','mid','premium','luxury'], right=False)

#D
diamonds['log_price'] = np.log(diamonds['price'])
#E
diamonds['heavy'] = diamonds['carat']>1.5

%timeit diamonds['tier'] = pd.cut(diamonds['price'], bins = [0,500,2500,10000,np.inf], labels=['budget','mid','premium','luxury'], right=False)
%timeit diamonds['tier'] = diamonds['price'].apply(lambda x: 'budget' if x < 500 else 'mid' if x < 2500 else 'premium' if x < 10000 else 'luxury')


2.32 ms ± 305 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
13.1 ms ± 1.73 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


---
## Level 2 — mpg: open questions

Load `sns.load_dataset('mpg')`. Drop nulls. No `.apply()`.

1. What fraction of cars have both above-median `mpg` and above-median `horsepower`? One expression.
2. Add `efficiency` = `mpg / weight * 1000` vectorized. Which `origin` has the highest mean `efficiency`?
3. Use `np.select()` to add `era`: `'70s'` (model_year < 75), `'mid'` (75–79), `'80s'` (80+).
4. Convert `origin` and `era` to `category`. How much memory does the DataFrame use before vs after (in KB)?
5. Is `np.corrcoef` between `weight` and `mpg` stronger or weaker than between `horsepower` and `mpg`? Print both.

In [ ]:
# Your code here
mpg = sns.load_dataset('mpg').dropna()

((mpg['mpg']>mpg['mpg'].median())&(mpg['horsepower']>mpg['horsepower'].median())).mean()

mpg['efficiency'] = mpg['mpg']/mpg['weight']*1000
mpg.groupby('origin')['efficiency'].mean().idxmax()

conds = [mpg['model_year']<75, mpg['model_year']<80]
chos = ['70s', 'mid']
mpg['era'] = np.select(conds, chos, default='80s')
m=mpg.memory_usage(deep=True).sum()/1024
print(f'memory before is {m:.2f}')
mpg[['origin', 'era']] = mpg[['origin', 'era']].astype('category')
m=mpg.memory_usage(deep=True,).sum()/1024
print(f'memory after is {m:.2f}')

cor1 = np.corrcoef(mpg['weight'], mpg['mpg'])[0,1]
cor2 = np.corrcoef(mpg['horsepower'], mpg['mpg'])[0,1]
print(f'{cor1:.2f}') ### negtative correlation is stronger
print(f'{cor2:.2f}')




memory before is 101.85
memory after is 56.89
-0.83
-0.78


---
## Level 3 — titanic + penguins: two mini-analyses

**Part A — titanic**

No `.apply()`. Build these columns vectorized, then answer the question.

- `ticket_class`: `'first'` (pclass == 1), `'second'` (pclass == 2), `'third'` (pclass == 3) — use `np.select()`
- `paid_premium`: True where fare > the 75th percentile of fare (`np.percentile`)
- Among passengers who `paid_premium`, what is the survival rate by `ticket_class`? Use groupby.
- Use a set comprehension to find all unique `embark_town` values where survival rate > 0.5.

**Part B — penguins**

Load `sns.load_dataset('penguins')`. Drop nulls.

- Add `bill_area` = `bill_length_mm * bill_depth_mm` vectorized.
- Use `transform` to add species-level mean and std of `bill_area`. Then add `bill_z` = `(bill_area - species mean) / species std` — a z-score within each species.
- Which species has the most penguins with `abs(bill_z) > 1` (outliers within their species)? Use groupby on a boolean column.

In [ ]:
# Your code here

titanic = sns.load_dataset('titanic')
conds  = [titanic['pclass']==1, titanic['pclass'] ==2 ]
chos = ['first','second']
titanic['ticket_class'] = np.select(conds, chos, default = 'third')


titanic['paid_premium'] = titanic['fare']>np.percentile(titanic['fare'], 75)

titanic[titanic['paid_premium']].groupby('ticket_class')['survived'].mean()

#{et for et, surv in zip(titanic['embark_town'], titanic['survivied'])}
rates = titanic.groupby('embark_town')['survived'].mean()
{et for et, r in zip(rates.index, rates) if r > 0.5}

#B
penguins = sns.load_dataset('penguins').dropna()

penguins['bill_area'] = penguins['bill_length_mm']*penguins['bill_depth_mm']

penguins['bm'] = penguins.groupby('species')['bill_area'].transform('mean')
penguins['bs'] = penguins.groupby('species')['bill_area'].transform('std')
penguins['bill_z'] = (penguins['bill_area']-penguins['bm'])/penguins['bs']

penguins['isOut'] = penguins['bill_z'].abs()>1
penguins.groupby('species')['isOut'].count().idxmax()




'Adelie'

In [53]:
rates = titanic.groupby('embark_town')['survived'].mean()
{et for et, r in zip(rates.index, rates) if r > 0.5}

{'Cherbourg'}

---
## Level 4 — df pipeline

Write a `.pipe()` chain on the df data below. No `.apply()`.

- **`prepare(df)`** — convert `category` and `warehouse` to `category`; add `total_value` (stock × unit_price) and `reorder_gap` (stock - reorder_point) vectorized
- **`classify(df)`** — add `stock_status` with `np.select()`: `'critical'` (reorder_gap < 0), `'low'` (reorder_gap < 20), `'ok'` otherwise. Add `high_value`: True where `total_value` is above the 75th percentile.
- **`report(df)`** — return a pivot table: mean `total_value` by `warehouse` × `stock_status`, `observed=True`, `fill_value=0`

After the chain:
- Stack the pivot table. What is the `(warehouse, stock_status)` combination with the highest mean total value?
- Use a list comprehension with `zip` to collect product names where `stock_status == 'critical'`.
- What fraction of `high_value` items are also `'critical'`? Use `.loc` and `np.mean()`.

In [52]:
inventory= pd.DataFrame({
    'product':       ['Laptop','Mouse','Keyboard','Monitor','Headset','Webcam',
                      'Desk','Chair','Lamp','Cables','Tablet','Charger'],
    'category':      ['Electronics','Electronics','Electronics','Electronics','Electronics','Electronics',
                      'Furniture','Furniture','Furniture','Accessories','Electronics','Accessories'],
    'warehouse':     ['A','A','A','B','B','B','A','A','B','B','A','B'],
    'stock':         [12, 85, 40, 5, 22, 60, 8, 3, 30, 150, 18, 200],
    'reorder_point': [10, 50, 30, 10, 20, 40, 5, 5, 15, 100, 15, 100],
    'unit_price':    [999, 25, 75, 349, 89, 120, 450, 320, 60, 8, 599, 35]
})

# Your code here


def prepare(df):
    df[['category','warehouse']] = df[['category','warehouse']].astype('category')
    df['total_value'] = df['stock']* df['unit_price']
    df['reorder_gap'] = df['stock'] - df['reorder_point']
    return df

def classify(df):
   conds = [df['reorder_gap']<0, df['reorder_gap']<20]
   chois = ['critical','low']
   df['stock_status'] = np.select(conds, chois, default='ok')
   df['high_value'] = df['total_value']>np.percentile(df['total_value'],75)
   return df 

def report(df):
    p = df.pivot_table(
        index = 'warehouse',
        columns = 'stock_status',
        values = 'total_value',
        observed = True, 
        fill_value = 0
    )
    return p

res = inventory.copy().pipe(prepare).pipe(classify).pipe(report)
sres = res.stack()
sres.idxmax()

ss = inventory.copy().pipe(prepare).pipe(classify)

[name for name, sts in zip(ss['product'], ss['stock_status']) if sts == 'critical']

np.mean(ss.loc[ss['high_value'],'stock_status']=='critical')



np.float64(0.0)

After the chain:
- Stack the pivot table. What is the `(warehouse, stock_status)` combination with the highest mean total value?
- Use a list comprehension with `zip` to collect product names where `stock_status == 'critical'`.
- What fraction of `high_value` items are also `'critical'`? Use `.loc` and `np.mean()`.

In [51]:
ss

,product,category,warehouse,stock,reorder_point,unit_price,total_value,reorder_gap,stock_status,high_value
0,Laptop,Electronics,A,12,10,999,11988,2,low,True
1,Mouse,Electronics,A,85,50,25,2125,35,ok,False
2,Keyboard,Electronics,A,40,30,75,3000,10,low,False
3,Monitor,Electronics,B,5,10,349,1745,-5,critical,False
4,Headset,Electronics,B,22,20,89,1958,2,low,False
5,Webcam,Electronics,B,60,40,120,7200,20,ok,True
6,Desk,Furniture,A,8,5,450,3600,3,low,False
7,Chair,Furniture,A,3,5,320,960,-2,critical,False
8,Lamp,Furniture,B,30,15,60,1800,15,low,False
9,Cables,Accessories,B,150,100,8,1200,50,ok,False
